In [17]:
import pandas as pd
import numpy as np


df_products = pd.read_excel(
    "data/products.xlsx",
    sheet_name="products"
)

df_categories = pd.read_excel(
    "data/products.xlsx",
    sheet_name="categories"
)

df_prod_full = df_products.merge(
    df_categories[["category_id", "category_name", "description"]],
    on="category_id",
    how="left",
    suffixes=("_product", "_category")
)

df_prod_full = df_prod_full.rename(
    columns={"category_name_category": "top_category"}
)

df_events_records = pd.read_json(
    "data/events_records.json",
    orient="records"
)

df_events_lines = pd.read_json(
    "data/events_lines.json",
    lines=True
)

if "event_id" not in df_events_records.columns:
    df_events_records = df_events_records.T
    df_events_records.columns = [
        "event_id",
        "user_id",
        "event_time",
        "event_type",
        "page"
    ]


df_events = pd.concat(
    [df_events_records, df_events_lines],
    ignore_index=True
)

df_events["event_time"] = pd.to_datetime(
    df_events["event_time"],
    errors="coerce"
)

df_events["product_id"] = pd.to_numeric(
    df_events["page"]
    .astype("string")
    .str.extract(r"/product/(\d+)")[0],
    errors="coerce"
)

df_events = df_events.merge(
    df_prod_full[
        ["product_id", "category_id", "top_category"]
    ],
    on="product_id",
    how="left"
)

print("Товары с категориями:")
print(df_prod_full.head())

print("\nСобытия с категориями:")
print(df_events.head())

print("\nКоличество строк и столбцов событий:")
print(df_events.shape)

print("\nКоличество пропусков:")
print(df_events.isna().sum())

Товары с категориями:
   product_id product_name  category_id category_name_product  weight_kg  \
0           1    Product 1            1           Electronics      11.55   
1           2    Product 2            3                 Sport       2.22   
2           3    Product 3            5                  Toys      11.43   
3           4    Product 4            1           Electronics       4.17   
4           5    Product 5            5                  Toys       1.81   

  top_category                  description  
0  Electronics  Description for Electronics  
1        Sport        Description for Sport  
2         Toys         Description for Toys  
3  Electronics  Description for Electronics  
4         Toys         Description for Toys  

События с категориями:
  event_id user_id          event_time   event_type         page  product_id  \
0        1    1085 2024-01-30 20:37:32    page_view  /product/26          26   
1        2    1006 2024-02-24 22:55:13  add_to_cart  /product

In [18]:
# Задание 1
purchase_pivot = pd.crosstab(
    df_events["top_category"],
    df_events["event_type"] == "purchase"
)

purchase_pivot = purchase_pivot.rename(
    columns={
        False: "Не purchase",
        True: "Purchase"
    }
)

print(purchase_pivot)

event_type    Не purchase  Purchase
top_category                       
Electronics           454       108
Fashion               312        82
Home                  232        58
Sport                 572       138
Toys                  678       198


In [19]:
# Задание 2
purchase_stats = df_events.dropna(
    subset=["top_category"]
).groupby("top_category").agg(
    all_events=("event_type", "count"),
    purchase_count=(
        "event_type",
        lambda x: (x == "purchase").sum()
    )
)

purchase_stats["purchase_share"] = (
    purchase_stats["purchase_count"]
    / purchase_stats["all_events"]
    * 100
)

purchase_stats = purchase_stats.sort_values(
    "purchase_share",
    ascending=False
)

print(purchase_stats.round(2))

              all_events  purchase_count  purchase_share
top_category                                            
Toys                 876             198           22.60
Fashion              394              82           20.81
Home                 290              58           20.00
Sport                710             138           19.44
Electronics          562             108           19.22


In [20]:
# Задание 3
top_category_name = purchase_stats[
    "purchase_share"
].idxmax()

df_sessions = df_events.dropna(
    subset=["user_id", "event_time"]
).copy()

df_sessions = df_sessions.sort_values(
    ["user_id", "event_time"]
)

df_sessions["time_diff"] = (
    df_sessions.groupby("user_id")["event_time"].diff()
)

df_sessions["new_session"] = (
    df_sessions["time_diff"].isna()
    | (df_sessions["time_diff"] > pd.Timedelta(minutes=30))
)

df_sessions["session_id"] = (
    df_sessions.groupby("user_id")["new_session"].cumsum()
)

df_sessions["target_purchase"] = (
    (df_sessions["event_type"] == "purchase")
    & (df_sessions["top_category"] == top_category_name)
)

session_stats = df_sessions.groupby(
    ["user_id", "session_id"]
).agg(
    has_purchase=("target_purchase", "any")
)

conversion_top_category = (
    session_stats.groupby("user_id")["has_purchase"]
    .mean()
    .to_frame("conversion_top_category")
)

print(conversion_top_category.head())

         conversion_top_category
user_id                         
1001                    0.040000
1002                    0.125000
1003                    0.128205
1004                    0.171429
1005                    0.090909
